# Ejercicio 3 - Consultas directas sobre archivos Parquet

Todas las consultas viven en `sql/` y se ejecutan con `labutil.q`, que imprime el SQL usado.
Las vistas de `sql/00_views.sql` solo envuelven `read_parquet(...)`: **no hay importacion previa a una tabla**.

In [1]:
import sys; sys.path.insert(0, "../scripts")
import labutil as lu
con = lu.connect(anio="2026")  # alcance de este ejercicio: solo 2026 (use anio="*" para todos los anios)

## 3.1 Cantidad de archivos
**Objetivo:** confirmar que estan los 8 meses publicados de cada taxi. **Fuente:** `data/raw/*/*/*.parquet`.

In [2]:
lu.q(con, "03_01_files.sql")

-- 03_01_files.sql
-- Objetivo: contar y listar los archivos Parquet disponibles (3.1).
-- Fuente: data/raw/*/*/*.parquet
SELECT regexp_extract(file, '/(yellow|green)/', 1) AS taxi, count(*) AS archivos
FROM glob('/Users/a87009/Documents/Año 4 Semestre 2/Data Science/Laboratorio 8 DuckDB/lab8-duckdb/data/raw/*/2026/*.parquet') AS t(file)
GROUP BY ALL ORDER BY taxi;



,taxi,archivos
0,green,8
1,yellow,8


**Resultado / decision:** 8 archivos yellow + 8 green (ene-ago 2026), coincide con `verify_data.py`. No se necesita descargar nada mas.

## 3.2 Cantidad de registros
**Objetivo:** filas por archivo desde los metadatos del footer (no lee datos) y total por taxi. **Fuente:** todos los Parquet.

In [3]:
lu.q(con, "03_02_rows_per_file.sql")

-- 03_02_rows_per_file.sql
-- Objetivo: filas por archivo leyendo solo los metadatos del footer (3.2).
-- Fuente: todos los Parquet de yellow y green.
SELECT regexp_extract(file_name, '([a-z]+)_tripdata_(\d{4}-\d{2})', 1) AS taxi,
       regexp_extract(file_name, '_(\d{4}-\d{2})', 1) AS mes,
       num_rows, num_row_groups
FROM parquet_file_metadata('/Users/a87009/Documents/Año 4 Semestre 2/Data Science/Laboratorio 8 DuckDB/lab8-duckdb/data/raw/*/2026/*.parquet')
ORDER BY taxi, mes;



,taxi,mes,num_rows,num_row_groups
0,green,2026-01,40272,1
1,green,2026-02,37373,1
2,green,2026-03,44208,1
3,green,2026-04,44238,1
4,green,2026-05,44921,1
5,green,2026-06,44163,1
6,green,2026-07,41252,1
7,green,2026-08,40687,1
8,yellow,2026-01,3724889,4
9,yellow,2026-02,3399866,4


In [4]:
lu.q(con, "03_03_total_rows.sql")

-- 03_03_total_rows.sql
-- Objetivo: total de registros por tipo de taxi (3.2).
-- Fuente: yellow_tripdata_*.parquet y green_tripdata_*.parquet (via vista trips).
SELECT taxi, count(*) AS registros FROM trips GROUP BY taxi
UNION ALL
SELECT 'TOTAL', count(*) FROM trips;



,taxi,registros
0,green,337114
1,yellow,29703355
2,TOTAL,30040469


**Resultado / decision:** ~30.0 millones de viajes (29.7 M yellow, 0.34 M green). Yellow supera por ~88x a green, por lo que
las comparaciones entre taxis deberan usar proporciones y no conteos absolutos. Los archivos yellow tienen 4 row groups, lo que
permite a DuckDB paralelizar la lectura.

## 3.3 y 3.4 Columnas y tipos de datos
**Fuente:** `yellow_tripdata_*` y `green_tripdata_*`.

In [5]:
lu.q(con, "03_04_schema_yellow.sql")

-- 03_04_schema_yellow.sql
-- Objetivo: columnas y tipos de los archivos yellow (3.3, 3.4).
-- Fuente: data/raw/yellow/*/*.parquet
DESCRIBE SELECT * FROM read_parquet('/Users/a87009/Documents/Año 4 Semestre 2/Data Science/Laboratorio 8 DuckDB/lab8-duckdb/data/raw/yellow/2026/*.parquet', union_by_name = true);



,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


In [6]:
lu.q(con, "03_05_schema_green.sql")

-- 03_05_schema_green.sql
-- Objetivo: columnas y tipos de los archivos green (3.3, 3.4).
-- Fuente: data/raw/green/*/*.parquet
DESCRIBE SELECT * FROM read_parquet('/Users/a87009/Documents/Año 4 Semestre 2/Data Science/Laboratorio 8 DuckDB/lab8-duckdb/data/raw/green/2026/*.parquet', union_by_name = true);



,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,lpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,lpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,store_and_fwd_flag,VARCHAR,YES,None,None,None
4,RatecodeID,BIGINT,YES,None,None,None
5,PULocationID,INTEGER,YES,None,None,None
6,DOLocationID,INTEGER,YES,None,None,None
7,passenger_count,BIGINT,YES,None,None,None
8,trip_distance,DOUBLE,YES,None,None,None
9,fare_amount,DOUBLE,YES,None,None,None


**Resultado / decision:** ambos comparten casi todo el esquema. Diferencias: yellow usa `tpep_*` y tiene `Airport_fee`; green usa `lpep_*`
y tiene `ehail_fee` y `trip_type`. Ambos incluyen `cbd_congestion_fee` (cargo de congestion de Manhattan) y `request_source`.
`passenger_count`, `RatecodeID` y `payment_type` vienen como `BIGINT` (nullables) en vez de enteros pequenos.
Decision: la vista `trips` renombra las columnas de fecha a `pickup`/`dropoff` y unifica ambos esquemas con `UNION ALL BY NAME`.

## 3.5 Muestra de registros
**Fuente:** vista `trips` (Parquet).

In [7]:
lu.q(con, "03_06_sample.sql")

-- 03_06_sample.sql
-- Objetivo: muestra aleatoria de 5 registros de cada taxi (3.5).
-- Fuente: vista trips (Parquet yellow + green).
(SELECT * FROM trips USING SAMPLE 5 ROWS)
UNION ALL BY NAME
(SELECT * FROM trips WHERE taxi = 'green' USING SAMPLE 5 ROWS);



,taxi,vendor_id,pickup,dropoff,passenger_count,trip_distance,ratecode_id,store_and_fwd_flag,pu_location,do_location,...,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,trip_type,data_year,data_month
0,yellow,2,2026-02-21 22:34:11,2026-02-21 22:56:07,<NA>,7.92,<NA>,None,48,243,...,0.0,0.0,1.0,57.07,NaN,NaN,0.75,<NA>,2026,2
1,yellow,2,2026-02-21 20:08:35,2026-02-21 20:25:13,<NA>,4.86,<NA>,None,87,162,...,0.0,0.0,1.0,30.00,NaN,NaN,0.75,<NA>,2026,2
2,yellow,2,2026-02-21 23:47:31,2026-02-22 00:01:06,<NA>,3.49,<NA>,None,48,238,...,0.0,0.0,1.0,29.93,NaN,NaN,0.75,<NA>,2026,2
3,yellow,2,2026-02-21 20:03:11,2026-02-21 20:18:46,<NA>,2.39,<NA>,None,141,143,...,0.0,0.0,1.0,28.54,NaN,NaN,0.75,<NA>,2026,2
4,yellow,2,2026-02-21 23:40:36,2026-02-22 00:05:54,<NA>,5.86,<NA>,None,145,238,...,0.0,0.0,1.0,38.95,NaN,NaN,0.00,<NA>,2026,2


**Resultado / decision:** en yellow se ven filas con `passenger_count`, `ratecode_id` y `store_and_fwd_flag` nulos y `payment_type = 0`
(viajes de flujo no estandar, p. ej. reservados por app). Se tratan como categoria propia ("desconocido") y no se descartan.

## 3.6 Problemas de calidad de datos
**Objetivo:** cuantificar anomalias. **Fuente:** vista `trips`.

In [8]:
lu.q(con, "03_07_quality.sql").T

-- 03_07_quality.sql
-- Objetivo: contar anomalias de calidad en yellow y green (3.6).
-- Fuente: yellow_tripdata_*.parquet y green_tripdata_*.parquet (via vista trips).
SELECT taxi,
  count(*)                                                AS total,
  count(*) FILTER (WHERE passenger_count IS NULL)         AS pasajeros_nulos,
  count(*) FILTER (WHERE passenger_count = 0)             AS pasajeros_cero,
  count(*) FILTER (WHERE trip_distance = 0)               AS distancia_cero,
  count(*) FILTER (WHERE trip_distance >= 100)            AS distancia_100_o_mas,
  count(*) FILTER (WHERE fare_amount <= 0)                AS tarifa_no_positiva,
  count(*) FILTER (WHERE total_amount <= 0)               AS total_no_positivo,
  count(*) FILTER (WHERE total_amount >= 1000)            AS total_1000_o_mas,
  count(*) FILTER (WHERE dropoff <= pickup)               AS bajada_antes_o_igual_subida,
  count(*) FILTER (WHERE dropoff - pickup >= INTERVAL 6 HOUR) AS duracion_6h_o_mas,
  count(*) FILTER (WH

,0,1
taxi,green,yellow
total,337114,29703355
pasajeros_nulos,48775,7716688
pasajeros_cero,4527,91359
distancia_cero,12212,952231
distancia_100_o_mas,72,1224
tarifa_no_positiva,5879,179751
total_no_positivo,1566,167093
total_1000_o_mas,1,54
bajada_antes_o_igual_subida,234,371683


In [9]:
lu.q(con, "03_08_dates_out_of_range.sql")

-- 03_08_dates_out_of_range.sql
-- Objetivo: ver las fechas de recogida fuera del periodo del archivo (3.6).
-- Fuente: vista trips.
SELECT taxi, year(pickup) AS anio_pickup, data_year, count(*) AS viajes
FROM trips
WHERE year(pickup) <> data_year OR month(pickup) <> data_month
GROUP BY ALL ORDER BY viajes DESC LIMIT 15;



,taxi,anio_pickup,data_year,viajes
0,yellow,2026,2026,129
1,green,2026,2026,84
2,yellow,2009,2026,6
3,yellow,2025,2026,6
4,green,2009,2026,5
5,green,2008,2026,5
6,green,2025,2026,4
7,yellow,2008,2026,4
8,yellow,2001,2026,1


**Problemas identificados**

1. **Fechas fuera del periodo del archivo:** viajes con pickup en 2001, 2008, 2009 o 2025 dentro de archivos de 2026 (relojes mal configurados). Tambien hay pickups del mes siguiente/anterior en el limite del mes.
2. **Duracion no positiva:** ~372 mil viajes yellow tienen `dropoff <= pickup`, y otros miles duran 6 horas o mas.
3. **Distancia 0** en ~952 mil viajes yellow, y distancias >= 100 millas (~1.2 mil).
4. **Tarifas / totales <= 0** (~180 mil y ~167 mil en yellow): reembolsos, disputas o viajes cancelados.
5. **Valores nulos:** ~7.7 M de filas yellow (26%) sin `passenger_count`, `ratecode_id` ni `store_and_fwd_flag`, todas con `payment_type = 0`.
6. **Totales extremos** (>= 1000) y propinas negativas.

**Decision:** definir `trips_clean` (en `00_views.sql`) que exige pickup dentro del mes del archivo, duracion entre 0 y 6 h,
distancia entre 0 y 100 millas, tarifa > 0 y total entre 0 y 1000. Las consultas de conteo global usan `trips`; las de
promedios, distribuciones y KPIs usan `trips_clean`.

In [10]:
lu.q(con, "03_09_clean_summary.sql")

-- 03_09_clean_summary.sql
-- Objetivo: cuantificar el efecto del filtro trips_clean (decision de 3.6).
-- Fuente: vistas trips y trips_clean.
SELECT t.taxi, t.total, c.validos, t.total - c.validos AS descartados,
       round(100.0 * (t.total - c.validos) / t.total, 2) AS pct_descartado
FROM (SELECT taxi, count(*) total FROM trips GROUP BY taxi) t
JOIN (SELECT taxi, count(*) validos FROM trips_clean GROUP BY taxi) c USING (taxi);



,taxi,total,validos,descartados,pct_descartado
0,yellow,29703355,28221859,1481496,4.99
1,green,337114,317987,19127,5.67


Se descarta ~5% de viajes en ambos taxis, un costo razonable para evitar que valores absurdos distorsionen medias y percentiles.

## 3.9 Que significa consultar directamente un Parquet

`read_parquet('ruta/*.parquet')` convierte los archivos en una tabla virtual: DuckDB lee el *footer* (esquema, estadisticas por
row group) y solo despues lee los bytes necesarios. Por ser **columnar**, solo se leen las columnas referenciadas
(*projection pushdown*), y con las estadisticas min/max se saltan row groups completos que no cumplen el filtro
(*predicate pushdown*). No hay fase de importacion, no hay copia de los datos y no hace falta un servidor.
Con volumenes grandes esto ahorra tiempo (se consulta de inmediato), disco (no se duplica) y memoria (no se carga todo en RAM
como en pandas), y los datos nuevos aparecen con solo agregar archivos que coincidan con el glob.